# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv("customers.csv")

print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())


   customer_id  age  income  spending region subscribed
0            1   25    61.3      39.1   West         No
1            2   25    54.9      42.1  South        Yes
2            3   51   105.1      59.9  South         No
3            4   39     NaN      49.9  South         No
4            5   43    72.4      48.3   East         No

Data types:
customer_id      int64
age              int64
income         float64
spending       float64
region             str
subscribed         str
dtype: object

Missing values:
customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [ ]:
# 1 Standardize the region labels
df["region"] = df["region"].str.strip().str.title()

# 2 Treat impossible negative income as missing
df["income"] = pd.to_numeric(df["income"], errors="coerce")
df.loc[df["income"] < 0, "income"] = np.nan

# 3 Fill missing income values with the median
income_median = df["income"].median()
df["income"] = df["income"].fillna(income_median)

print(df.isna().sum())
print("\nRegion counts:")
print(df["region"].value_counts())


customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64

Region counts:
region
East     21
West     10
South     9
Name: count, dtype: int64


The dataset had three data-quality issues. Two income values were missing, the region label east was written in the wrong case, and one income value was impossible for the variable (-50), so I converted it to missing before filling the missing income values with the median (84.7). I used the median because it is more resistant than the mean to skewed data and outliers.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [ ]:
corr = df["income"].corr(df["spending"])
print(f"Pearson correlation r = {corr:.3f}")


: 

Yes - customers with higher incomes also tend to spend more, with a strong positive correlation (r about 0.72).


### Q2. Is a customer's region related to whether they subscribed?

In [ ]:
contingency = pd.crosstab(df["region"], df["subscribed"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(contingency)
print(f"chi-square = {chi2:.3f}, p-value = {p_value:.4f}")
print("\nExpected counts:\n", expected)


Yes - region and subscription status are related (chi-square = 13.16, p = 0.0014), so subscription rates differ by region.


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [ ]:
X = sm.add_constant(df[["income", "age"]])
y = df["spending"]
model = sm.OLS(y, X).fit()
print(f"R-squared = {model.rsquared:.3f}")
print(model.summary())


Together, income and age explain about 52% of the variation in spending (R-squared ~0.52), and income is a significant predictor (p = 0.003) while age is not (p = 0.515).


### Q4. Which region has the most customers, and what is the average income overall?

In [ ]:
region_counts = df["region"].value_counts()
overall_income_mean = df["income"].mean()
print(region_counts)
print(f"Overall mean income = {overall_income_mean:.2f}")


East has the most customers (21), and the overall average income is 86.46.


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** The choice of test depended on the variables and the question: correlation for continuous-continuous relationships, chi-square for categorical-categorical relationships, multiple regression for one continuous outcome with several predictors, and descriptive statistics for single-variable summaries.

**b)** No, we cannot say that higher income causes higher spending from this analysis alone because correlation does not prove that it caused it. Other variables such as age, preferences, or region could influence both income and spending.


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [ ]:
contingency = pd.crosstab(df["region"], df["subscribed"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(expected)
print("All expected counts >= 5:", (expected >= 5).all())


No - one expected count is 4.5, which is below 5, so the chi-square assumption is not fully met. I would tell the company to treat the result as suggestive but not fully reliable unless we combine categories or collect more data.


### Push your code to GitHub